When you extract frequent itemsets (words or sets of words that appear together often across your sentences), you reveal underlying structure:

Sentences that share many frequent itemsets are likely to be similar—for example, about the same topic or intent.

This means your sentences can be grouped (clustered) based on how much they “share” these itemsets.

# André

In [27]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Data

In [114]:
data = pd.read_csv('../data/horoscope_full.csv')

data.columns
data = data.head(1000)

# this dataset is not real is generated for demonstration purposes only

In [115]:
data.rename(columns={'horoscope': 'token'}, inplace=True)

In [116]:
#for i in range(len(data)):
    #data["token"][i] = "a a a"

In [117]:
import re

def clean_keep_apostrophe(text):
    # Only keep letters, digits, apostrophes, and whitespace
    return re.sub(r"[^\w\d'\s]", "", text)

# Apply to your DataFrame
data['token'] = data['token'].apply(clean_keep_apostrophe)
data['token'] = data['token'].str.split()

data.head()

,ID,sign,category,date,token
0,1,aries,general,20200617,"[There's, a, great, day, ahead, of, you, Aries..."
1,2,aries,general,20200618,"[People, will, understand, and, appreciate, yo..."
2,3,aries,general,20200619,"[You, are, very, interested, in, technological..."
3,4,aries,general,20200620,"[Stress, from, overwork, could, have, you, fee..."
4,5,aries,general,20200621,"[This, is, a, good, day, to, stand, up, for, y..."


In [118]:
data["original_token_size"] = data["token"].apply(len)
data["original_token_size"].describe()

count    1000.000000
mean       60.542000
std        10.350815
min        36.000000
25%        49.000000
50%        65.000000
75%        68.000000
max        87.000000
Name: original_token_size, dtype: float64

In [119]:
data["token"] = data["token"].apply(lambda x: [w.lower() for w in x])
data


,ID,sign,category,date,token,original_token_size
0,1,aries,general,20200617,"[there's, a, great, day, ahead, of, you, aries...",61
1,2,aries,general,20200618,"[people, will, understand, and, appreciate, yo...",71
2,3,aries,general,20200619,"[you, are, very, interested, in, technological...",52
3,4,aries,general,20200620,"[stress, from, overwork, could, have, you, fee...",73
4,5,aries,general,20200621,"[this, is, a, good, day, to, stand, up, for, y...",65
...,...,...,...,...,...,...
995,996,aries,career,20210307,"[the, week, is, off, to, a, slow, start, for, ...",48
996,997,aries,career,20210308,"[don't, be, afraid, to, let, your, sensitive, ...",46
997,998,aries,career,20210309,"[the, burst, of, energy, that, you, have, been...",47
998,999,aries,career,20210310,"[it, will, be, hard, to, say, no, to, anyone, ...",43


# Word Co-occurrence Similarity Matrix

In [120]:
all_words = set()
for tokens in data['token']:
    for word in tokens:
        all_words.add(word)

all_words_list = sorted(all_words)

In [121]:
word_counter = {word: 0 for word in all_words_list}
for tokens in data['token']:
    for word in tokens:
        word_counter[word] += 1

word_counter

{'12': 1,
 '20minute': 1,
 'a': 1616,
 'abandon': 2,
 'abilities': 11,
 'ability': 23,
 'able': 52,
 'aboard': 3,
 'about': 249,
 'above': 5,
 'abrasive': 1,
 'abruptly': 1,
 'absence': 1,
 'absolutely': 2,
 'absolutelynoescape': 1,
 'absorb': 1,
 'absorption': 1,
 'absurd': 1,
 'abusing': 1,
 'academia': 1,
 'accept': 7,
 'acceptable': 1,
 'accepting': 1,
 'access': 2,
 'accident': 1,
 'acclimatize': 1,
 'accomplish': 20,
 'accomplished': 1,
 'accomplishment': 2,
 'accomplishments': 3,
 'according': 1,
 'accordingly': 2,
 'account': 4,
 'accuracy': 1,
 'accurate': 3,
 'achieve': 6,
 'achievements': 2,
 'achieves': 1,
 'acknowledge': 4,
 'acknowledgment': 2,
 'acquaintances': 4,
 'acquainted': 2,
 'across': 6,
 'act': 20,
 'acting': 1,
 'action': 24,
 'actions': 14,
 'active': 3,
 'actively': 4,
 'activities': 10,
 'activity': 6,
 'actual': 2,
 'actually': 20,
 'acuity': 2,
 'acupuncture': 1,
 'acute': 1,
 'ad': 1,
 'adapt': 3,
 'adaptability': 1,
 'adaptive': 1,
 'add': 13,
 'addicted

In [122]:
data["token_no_reps"] = data["token"].apply(set)

In [124]:
co_occurence_matrix = pd.DataFrame(0, index=all_words_list, columns=all_words_list)

# Count co-occurrences
for word in all_words_list:
    for sentence in data['token_no_reps']:
        # normalize sentence tokens to lowercase
        sentence_l = [w.lower() for w in sentence]
        if word in sentence_l:
            for co_word in sentence_l:
                co_occurence_matrix.at[word, co_word] += 1

co_occurence_matrix

,12,20minute,a,abandon,abilities,ability,able,aboard,about,above,...,your,youre,yours,yourself,yourselves,youve,yoyo,zany,zesty,zone
12,1,0,0,0,0,0,0,0,0,0,...,1,0,0,1,0,0,0,0,0,0
20minute,0,1,1,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
a,0,1,787,2,7,18,39,1,185,4,...,610,4,8,146,2,1,1,1,1,1
abandon,0,0,2,2,0,0,0,0,0,0,...,2,0,0,2,0,0,0,0,0,0
abilities,0,0,7,0,11,0,0,1,2,0,...,11,0,1,3,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
youve,0,0,1,0,0,0,0,0,1,0,...,1,0,0,0,0,1,0,0,0,0
yoyo,0,0,1,0,0,0,0,0,0,0,...,1,0,0,1,0,0,1,0,0,0
zany,0,0,1,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,1,0,0
zesty,0,0,1,0,0,1,0,0,1,0,...,1,0,0,0,0,0,0,0,1,0


# Jaccard Similarity Between Sentences

I guess that this is basically what Theresio has!

In [126]:
data

,ID,sign,category,date,token,original_token_size,token_no_reps
0,1,aries,general,20200617,"[there's, a, great, day, ahead, of, you, aries...",61,"{there's, issues, like, empathy, you, solve, f..."
1,2,aries,general,20200618,"[people, will, understand, and, appreciate, yo...",71,"{like, people, on, your, appreciate, sign, spo..."
2,3,aries,general,20200619,"[you, are, very, interested, in, technological...",52,"{issues, like, breakthroughs, interested, wher..."
3,4,aries,general,20200620,"[stress, from, overwork, could, have, you, fee...",73,"{needs, stress, could, your, usual, weaker, ma..."
4,5,aries,general,20200621,"[this, is, a, good, day, to, stand, up, for, y...",65,"{pushover, on, your, free, may, use, you, pick..."
...,...,...,...,...,...,...,...
995,996,aries,career,20210307,"[the, week, is, off, to, a, slow, start, for, ...",48,"{issues, on, your, caught, or, you, from, focu..."
996,997,aries,career,20210308,"[don't, be, afraid, to, let, your, sensitive, ...",46,"{workplace, people, your, know, you, human, af..."
997,998,aries,career,20210309,"[the, burst, of, energy, that, you, have, been...",47,"{finally, on, cool, burst, been, hoping, you, ..."
998,999,aries,career,20210310,"[it, will, be, hard, to, say, no, to, anyone, ...",43,"{anyone, your, hard, or, positive, tackle, you..."


In [127]:
jacard_similarity_matrix = pd.DataFrame(0.0, index=range(data.shape[0]), columns=range(data.shape[0]))

for i in range(data.shape[0]):
    for j in range(data.shape[0]):
        tokens_i = set(data.at[i, 'token'])
        tokens_j = set(data.at[j, 'token'])
        intersection = tokens_i.intersection(tokens_j)
        union = tokens_i.union(tokens_j)
        if len(union) > 0:
            jacard_similarity_matrix.at[i, j] = len(intersection) / len(union)
        else:
            jacard_similarity_matrix.at[i, j] = 0.0


In [128]:
jacard_similarity_matrix

,0,1,2,3,4,5,6,7,8,9,...,990,991,992,993,994,995,996,997,998,999
0,1.000000,0.166667,0.142857,0.123596,0.152941,0.185185,0.112360,0.136364,0.160920,0.155556,...,0.123288,0.108108,0.092105,0.135135,0.051282,0.144737,0.076923,0.131579,0.123288,0.106667
1,0.166667,1.000000,0.137931,0.099010,0.184783,0.188889,0.157895,0.180851,0.131313,0.173469,...,0.134146,0.134146,0.132530,0.117647,0.107143,0.126437,0.187500,0.141176,0.134146,0.132530
2,0.142857,0.137931,1.000000,0.080460,0.095238,0.111111,0.094118,0.105882,0.104651,0.101124,...,0.070423,0.070423,0.100000,0.068493,0.055556,0.140845,0.098592,0.111111,0.055556,0.100000
3,0.123596,0.099010,0.080460,1.000000,0.142857,0.133333,0.105263,0.191011,0.126316,0.145833,...,0.128205,0.100000,0.112500,0.084337,0.086420,0.094118,0.111111,0.095238,0.113924,0.098765
4,0.152941,0.184783,0.095238,0.142857,1.000000,0.234568,0.211765,0.195402,0.166667,0.186813,...,0.178082,0.131579,0.144737,0.173333,0.075000,0.151899,0.173333,0.200000,0.194444,0.101266
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,0.144737,0.126437,0.140845,0.094118,0.151899,0.141026,0.108434,0.162500,0.160494,0.141176,...,0.119403,0.153846,0.206349,0.149254,0.086957,1.000000,0.184615,0.128571,0.136364,0.134328
996,0.076923,0.187500,0.098592,0.111111,0.173333,0.146667,0.155844,0.184211,0.109756,0.132530,...,0.142857,0.161290,0.196721,0.121212,0.107692,0.184615,1.000000,0.151515,0.180328,0.196721
997,0.131579,0.141176,0.111111,0.095238,0.200000,0.205479,0.123457,0.108434,0.120482,0.129412,...,0.121212,0.156250,0.136364,0.169231,0.088235,0.128571,0.151515,1.000000,0.121212,0.102941
998,0.123288,0.134146,0.055556,0.113924,0.194444,0.183099,0.160000,0.128205,0.186667,0.121951,...,0.147541,0.147541,0.183333,0.200000,0.093750,0.136364,0.180328,0.121212,1.000000,0.126984


# Numeric Similarity Matrix

In [129]:
freq_itemsets_data = pd.read_csv('../pre_results/frequent_itemsets.csv')
freq_itemsets_data

,itemset,count,support,k
0,"('you',)",20801,0.947265,1
1,"('to',)",20456,0.931554,1
2,"('the',)",18779,0.855185,1
3,"('and',)",18738,0.853318,1
4,"('your',)",17633,0.802996,1
...,...,...,...,...
72066,"('a', 'great', 'month')",440,0.020037,3
72067,"('family', 'in', 'more')",440,0.020037,3
72068,"('all', 'friends', 'work')",440,0.020037,3
72069,"('to', 'world', 'year')",440,0.020037,3


In [131]:
freq_itemsets_df_k_equal_2 = freq_itemsets_data[freq_itemsets_data['k'] == 2]
freq_itemsets_df_k_equal_2

,itemset,count,support,k
451,"('to', 'you')",19368,0.882007,2
452,"('the', 'you')",17790,0.810146,2
453,"('and', 'you')",17749,0.808279,2
454,"('the', 'to')",17452,0.794754,2
455,"('and', 'to')",17441,0.794253,2
...,...,...,...,...
9212,"('it', 'special')",440,0.020037,2
9213,"('current', 'today')",440,0.020037,2
9214,"('will', 'would')",440,0.020037,2
9215,"('and', 'late')",440,0.020037,2


In [136]:
# Pairwise shared 2-itemsets (k=2)
# For each sentence, build the set of 2-itemsets (unordered pairs) it contains,
# then compute, for every pair (i, j), how many 2-itemsets they share.

from itertools import combinations
import numpy as np

# Ensure tokens are lists of lowercased words
data['token'] = data['token'].apply(lambda toks: [w.lower() for w in toks])

# Build set of 2-itemsets (frozenset) for each sentence
def two_itemsets_from_tokens(tokens):
    # use unique tokens per sentence to avoid duplicate pairs from repeated words
    uniq = sorted(set(tokens))
    if len(uniq) < 2:
        return set()
    return set(frozenset(pair) for pair in combinations(uniq, 2))

two_itemsets_per_sentence = [two_itemsets_from_tokens(t) for t in data['token']]

n = len(two_itemsets_per_sentence)
print(f"Number of sentences: {n}")

# Create an NxN matrix (symmetric) of shared 2-itemset counts
shared_matrix = np.zeros((n, n), dtype=int)
for i in range(n):
    si = two_itemsets_per_sentence[i]
    # small optimization: skip empty
    if not si:
        continue
    for j in range(i+1, n):
        sj = two_itemsets_per_sentence[j]
        if not sj:
            continue
        shared = si.intersection(sj)
        cnt = len(shared)
        if cnt:
            shared_matrix[i, j] = cnt
            shared_matrix[j, i] = cnt

# Wrap as DataFrame with sentence indices
shared_df = pd.DataFrame(shared_matrix, index=data.index, columns=data.index)

# Show an example: top sentence pairs with most shared 2-itemsets
pairs = []
for i in range(n):
    for j in range(i+1, n):
        if shared_matrix[i, j] > 0:
            pairs.append((i, j, shared_matrix[i, j]))

pairs_sorted = sorted(pairs, key=lambda x: x[2], reverse=True)

print(f"Number of sentence pairs sharing at least one 2-itemset: {len(pairs_sorted)}")
print('\nTop 20 pairs (i, j, shared_2_itemsets_count):')
for rec in pairs_sorted[:20]:
    i, j, cnt = rec
    print(i, j, cnt)
    # optional: show the sentences and the shared 2-itemsets
    shared_sets = two_itemsets_per_sentence[i].intersection(two_itemsets_per_sentence[j])
    print('  shared:', [tuple(sorted(list(s))) for s in shared_sets])
    print('  sentence i:', ' '.join(data.at[i, 'token']))
    print('  sentence j:', ' '.join(data.at[j, 'token']))
    print()

# Expose results
# shared_df: full NxN matrix of shared 2-itemset counts
# pairs_sorted: list of (i, j, count) sorted by count desc

# Optionally save results
# shared_df.to_csv('../data/shared_2_itemsets_matrix.csv')
# pd.DataFrame(pairs_sorted, columns=['i','j','count']).to_csv('../data/shared_2_itemsets_pairs.csv', index=False)


Number of sentences: 1000
Number of sentence pairs sharing at least one 2-itemset: 499499

Top 20 pairs (i, j, shared_2_itemsets_count):
54 247 1326
  shared: [('might', 'your'), ('bringing', 'express'), ('few', 'your'), ('surprised', 'understanding'), ('important', 'itself'), ('than', 'you'), ("don't", 'listen'), ("don't", 'practicality'), ('at', 'understanding'), ('understanding', 'want'), ('might', 'than'), ('at', 'of'), ('could', 'need'), ('could', 'satisfaction'), ('deep', 'to'), ("it's", 'should'), ('sympathetic', 'today'), ('could', 'should'), ('even', 'more'), ('affairs', 'those'), ('if', 'you'), ('at', 'today'), ('listen', 'more'), ('at', 'might'), ('might', 'want'), ('affairs', 'understanding'), ('point', 'today'), ('bringing', 'could'), ('concern', 'shed'), ('for', 'you'), ('important', 'today'), ('might', 'more'), ("others'", 'point'), ('surprised', 'to'), ('itself', 'you'), ("don't", "others'"), ('even', 'want'), ('feelings', "others'"), ('those', 'to'), ('aries', 'concern

In [135]:
shared_df

,0,1,2,3,4,5,6,7,8,9,...,990,991,992,993,994,995,996,997,998,999
0,0,105,55,55,78,105,45,66,91,91,...,36,28,21,45,6,55,15,45,36,28
1,105,0,66,45,136,136,105,136,78,136,...,55,55,55,45,36,55,105,66,55,55
2,55,66,0,21,28,36,28,36,36,36,...,10,10,21,10,6,45,21,28,6,21
3,55,45,21,0,78,66,45,136,66,91,...,45,28,36,21,21,28,36,28,36,28
4,78,136,28,78,0,171,153,136,105,136,...,78,45,55,78,15,66,78,105,91,28
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,55,55,45,28,66,55,36,78,78,66,...,28,45,78,45,15,0,66,36,36,36
996,15,105,21,36,78,55,66,91,36,55,...,36,45,66,28,21,66,0,45,55,66
997,45,66,28,28,105,105,45,36,45,55,...,28,45,36,55,15,36,45,0,28,21
998,36,55,6,36,91,78,66,45,91,45,...,36,36,55,66,15,36,55,28,0,28


In [137]:
# Transform `shared_df` (NxN) into long-form DataFrame with columns ID1, ID2, value
# `shared_df` is assumed to have sentence indices as both index and columns
# Create a stacked DataFrame, keep only upper-triangle pairs (ID1 < ID2) to avoid duplicates,
# and keep only pairs with value > 0 (shared 2-itemsets).
shared_pairs_all = shared_df.stack().reset_index()
shared_pairs_all.columns = ['ID1', 'ID2', 'value']
# Ensure IDs are numeric if possible
try:
    shared_pairs_all['ID1'] = shared_pairs_all['ID1'].astype(int)
    shared_pairs_all['ID2'] = shared_pairs_all['ID2'].astype(int)
except Exception:
    pass
# Keep only upper-triangle (avoid mirrored duplicates)
shared_pairs_upper = shared_pairs_all[shared_pairs_all['ID1'] < shared_pairs_all['ID2']].copy()
# Filter non-zero shared counts
shared_pairs_nonzero = shared_pairs_upper[shared_pairs_upper['value'] > 0].reset_index(drop=True)
# Show top rows
print(f'Number of non-zero pairs: {len(shared_pairs_nonzero)}')
shared_pairs_nonzero.head(20)
# Optionally save to CSV for later use
shared_pairs_nonzero.to_csv('../pre_results/shared_2_itemsets_pairs.csv', index=False)
# Also expose both full and non-zero DataFrames for further analysis
shared_pairs_all = shared_pairs_all
shared_pairs_upper = shared_pairs_upper
shared_pairs_nonzero = shared_pairs_nonzero

Number of non-zero pairs: 499499


In [138]:
shared_pairs_nonzero

,ID1,ID2,value
0,0,1,105
1,0,2,55
2,0,3,55
3,0,4,78
4,0,5,105
...,...,...,...
499494,996,998,55
499495,996,999,66
499496,997,998,28
499497,997,999,21
